In [ ]:
import json
import os
from pathlib import Path

import requests

In [ ]:
# query_path = Path("../spi/search_query/claude-opus-5/2026-08-25_14-26-57.json")
# query_path = Path(
#     "../spi/search_query/claude-opus-5/2026-08-25_14-25-36.json"
# )  # Faulty query
# query_path = Path("../spi/search_query/claude-opus-5/2026-08-25_14-23-46.json")
query_path = Path("../spi/search_query/gpt-5.6-sol/2026-08-25_14-28-05.json")

with open(query_path) as file:
    search_query = json.load(file)

In [ ]:
search = {
    "query": search_query,
}

In [ ]:
url = "https://api.coresignal.com/cdapi/v2/employee_multi_source/search/es_dsl"
payload = json.dumps(search)

headers = {
    "Content-Type": "application/json",
    "apikey": os.environ["CORESIGNAL_API_KEY"],
}

response = requests.request("POST", url, headers=headers, data=payload)
person_ids: list[int] = json.loads(response.text)

print(person_ids)

In [ ]:
if 485761352 in person_ids:
    print("Found 485761352")

In [ ]:
with open("../spi/coresiganl_ids.json", "r") as f:
    cached_persons = json.load(f)

In [ ]:
person = person_ids[2]
person

In [ ]:
if person in cached_persons:
    print(f"Person {person} already cached, skipping API call.")
else:
    url = f"https://api.coresignal.com/cdapi/v2/employee_multi_source/collect/{person}"

    headers = {
        "Content-Type": "application/json",
        "apikey": os.environ["CORESIGNAL_API_KEY"],
    }

    response = requests.request("GET", url, headers=headers).json()

    # Write the response to a JSON file
    output_file = (
        Path("..")
        / "spi"
        / "coresignal"
        / "employee_multi_source"
        / f"{response['full_name']}.json"
    )

    output_file.parent.mkdir(parents=True, exist_ok=True)
    with open(output_file, "w") as f:
        json.dump(response, f, indent=4)

    print(response["full_name"])

In [ ]:
cached_persons.append(person)
with open("../spi/coresiganl_ids.json", "w") as f:
    json.dump(sorted(set(cached_persons)), f, indent=4)